# 03 — Break the CI *(re-segmentation)* · *Rompe el IC (re-segmentación)*

### English
The rate and its confidence interval feel objective — but they rest on a choice
made *before* any statistics run: **where the segment boundaries are drawn.**
Redraw them and a "statistically significant" hotspot can appear out of nothing,
or a real one can dissolve. This is the *modifiable areal unit problem* (MAUP),
and it is the sharpest tool in the "how to lie" kit.

We work directly with the hotspot API the tests exercise —
`getis_ord_star`, `two_sided_p`, and the Benjamini-Hochberg FDR control
`benjamini_hochberg` (`tests/test_fdr.py`, `tests/test_hotspot.py`). We will:

1. reproduce the **baseline** significant cluster,
2. **manufacture** a hotspot by *splitting* a borderline segment, and
3. **dissolve** the real hotspot by *merging* the corridor into one unit.

The FDR correction protects against multiple-comparison luck — but, as you will
see, it **cannot** protect against a rigged unit of analysis.

### Español
La tasa y su intervalo de confianza parecen objetivos, pero descansan sobre una
decisión tomada *antes* de cualquier estadística: **dónde se trazan los límites
de los segmentos.** Vuelve a trazarlos y un punto caliente "estadísticamente
significativo" puede aparecer de la nada, o uno real puede disolverse. Esto es el
*problema de la unidad de área modificable* (MAUP).

Trabajamos directamente con la API de puntos calientes que ejercitan las pruebas
— `getis_ord_star`, `two_sided_p` y el control FDR de Benjamini-Hochberg
`benjamini_hochberg`. Vamos a: (1) reproducir el **grupo base** significativo,
(2) **fabricar** un punto caliente *dividiendo* un segmento limítrofe, y (3)
**disolver** el punto caliente real *fusionando* el corredor en una sola unidad.
La corrección FDR protege contra la suerte de comparaciones múltiples — pero
**no** puede proteger contra una unidad de análisis manipulada.

In [ ]:
from pathlib import Path

from nearmiss.config import load_config
from nearmiss.engine import build_analysis


def repo_root() -> Path:
    """Walk up from the kernel's working directory to the repo root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate the nearmiss repo root")


ROOT = repo_root()
config = load_config(ROOT / "config" / "davis-demo.toml")
bundle = build_analysis(config)

names = {s.id: s.name for s in bundle.segments}
# The twelve ANALYZED blocks — the ones that carry exposure and reports. Every
# other block in the synthetic Davis grid is published as no-exposure context.
analyzed = [s for s in bundle.result.segments if s.exposure_estimate is not None]

print("city:", config.city, "(synthetic demonstration data — not real reports)")
print("exposure coverage:", round(bundle.result.exposure_coverage * 100), "%")
print("analyzed segments:", len(analyzed))

### The hotspot engine, in miniature · *El motor de puntos calientes, en miniatura*
`run_gi` takes a `{segment_id: rate}` map and the matching centroids, computes a
Getis-Ord Gi\* z-score per segment, converts it to a two-sided p-value, and
applies the Benjamini-Hochberg FDR procedure at the configured `alpha`. A segment
is a "significant hotspot" only if it is FDR-rejected **and** its z is positive.
This is the same sequence `nearmiss.stats.analyze` runs internally. · *`run_gi`
toma un mapa `{segmento: tasa}` y los centroides, calcula el z de Gi\*, lo
convierte en un valor p bilateral y aplica el procedimiento FDR de
Benjamini-Hochberg. Un segmento es "punto caliente significativo" solo si el FDR
lo rechaza **y** su z es positivo.*

In [ ]:
from nearmiss.geometry import haversine_m, polyline_centroid
from nearmiss.stats.getis_ord import benjamini_hochberg, getis_ord_star, two_sided_p

BAND = config.gi_band_m
ALPHA = config.fdr_alpha
by_id = {s.id: s for s in bundle.segments}


def band_neighbors(centroid_map, band_m):
    """Straight-line-band neighborhoods for this exercise's hand-made segmentations.

    The real pipeline (FIX-02) derives neighborhoods from the street network
    (nearmiss.network.SegmentGraph); the invented split/merged segments below
    have no network geometry, so a centroid band stands in. The MAUP lesson is
    the same either way: the segmentation choice moves the statistics.
    """
    ids = list(centroid_map)
    return {
        a: {
            b
            for b in ids
            if b != a
            and haversine_m(
                centroid_map[a][0], centroid_map[a][1], centroid_map[b][0], centroid_map[b][1]
            )
            <= band_m
        }
        for a in ids
    }


def run_gi(rate_map, centroid_map):
    """Return (z, significant_set) for a segmentation, matching the pipeline's Gi*+FDR."""
    z = getis_ord_star(rate_map, band_neighbors(centroid_map, BAND))
    pvals = {sid: two_sided_p(zi) for sid, zi in z.items()}
    rejected = benjamini_hochberg(pvals, ALPHA)
    significant = {sid for sid in rejected if z[sid] > 0.0}
    return z, significant


# Baseline segmentation: every analyzed block with a rate, at its true centroid.
base_rate = {s.segment_id: s.rate for s in analyzed}
base_cent = {sid: polyline_centroid(by_id[sid].coords) for sid in base_rate}

z0, sig0 = run_gi(base_rate, base_cent)
print("baseline significant hotspots:", sorted(sig0))
print("seg-05 borderline z = %.2f, raw p = %.3f, FDR-significant = %s"
      % (z0["seg-05"], two_sided_p(z0["seg-05"]), "seg-05" in sig0))
# FIX-02 moved the pipeline itself to street-network neighborhoods (where the
# borderline corridor block clears the FDR bar); this exercise keeps a plain
# straight-line band so the re-drawn segments below need no street geometry —
# the MAUP lesson is identical. Sanity-check the band baseline instead: the
# planted corridor hotspot is recovered, and the borderline block is not (yet)
# significant.
assert "seg-06" in sig0 and "seg-05" not in sig0

### Exhibit A — the borderline segment · *Exhibit A — el segmento limítrofe*
`seg-05` ("5th St (B–C)") sits right on the edge: its raw two-sided p-value is
**below 0.05**, so an *uncorrected* test would call it a significant hotspot — but
Benjamini-Hochberg, correcting for the twelve comparisons, holds it back. That
restraint is exactly what we are about to defeat with geometry. · *`seg-05` está
justo en el borde: su valor p crudo es **menor a 0.05**, así que una prueba *sin
corregir* lo llamaría significativo — pero Benjamini-Hochberg lo retiene. Esa
contención es justo lo que vamos a vencer con geometría.*

### Manufacture a hotspot: *split* the borderline segment · *Fabricar un punto caliente: dividir el segmento limítrofe*
Imagine "refining" the map by cutting `seg-05` into two shorter blocks at the same
place — a perfectly ordinary re-segmentation choice. Each half inherits the same
high rate and sits essentially on top of the other, so Gi\* now sees a *tight
cluster of high-rate units* where there was one. The borderline segment crosses
the line into "significant." No new reports were collected. · *Imagina "refinar"
el mapa cortando `seg-05` en dos bloques más cortos en el mismo lugar. Cada mitad
hereda la misma tasa alta y queda prácticamente encima de la otra, así que Gi\*
ahora ve un grupo denso de unidades de tasa alta. El segmento limítrofe cruza a
"significativo". No se recolectó ningún reporte nuevo.*

In [ ]:
split_rate = {sid: r for sid, r in base_rate.items() if sid != "seg-05"}
split_cent = {sid: c for sid, c in base_cent.items() if sid != "seg-05"}
la, lo = base_cent["seg-05"]
for i in range(2):  # two co-located halves of the former seg-05
    split_rate[f"seg-05a{i}"] = base_rate["seg-05"]
    split_cent[f"seg-05a{i}"] = (la + i * 1e-6, lo + i * 1e-6)

z1, sig1 = run_gi(split_rate, split_cent)
manufactured = {sid for sid in sig1 if sid.startswith("seg-05")}
print("after splitting seg-05 into 2 blocks, significant:", sorted(sig1))
print("MANUFACTURED hotspot on the (former) borderline segment:", sorted(manufactured))
assert manufactured and "seg-05" not in sig0  # was NOT significant before the split

### Dissolve the real hotspot: *merge* the corridor · *Disolver el punto caliente real: fusionar el corredor*
Now the opposite move. Gi\* detects a hotspot because a high-rate unit is
*surrounded by* other high-rate units. Merge the genuine 5th St corridor
(`seg-02`, `seg-05`, `seg-06`, `seg-07`, `seg-10`) into a **single** averaged
block and those supporting neighbors vanish — the cluster has no neighborhood
left to be significant against. The real, statistically sound hotspot
**disappears** entirely, again without changing a single report. · *El
movimiento opuesto. Gi\* detecta un punto caliente porque una unidad de tasa
alta está *rodeada* de otras. Fusiona el corredor real de 5th St en un **único**
bloque promediado y esos vecinos de apoyo desaparecen. El punto caliente real y
estadísticamente sólido **desaparece** por completo.*

In [ ]:
corridor = ["seg-02", "seg-05", "seg-06", "seg-07", "seg-10"]
merge_rate = {sid: r for sid, r in base_rate.items() if sid not in corridor}
merge_cent = {sid: c for sid, c in base_cent.items() if sid not in corridor}
merge_rate["seg-corridor"] = sum(base_rate[s] for s in corridor) / len(corridor)
merge_cent["seg-corridor"] = (
    sum(base_cent[s][0] for s in corridor) / len(corridor),
    sum(base_cent[s][1] for s in corridor) / len(corridor),
)

z2, sig2 = run_gi(merge_rate, merge_cent)
print("after merging the 5th St corridor into one block, significant:", sorted(sig2))
assert "seg-06" in sig0 and not sig2  # the real hotspot dissolved: no significant cluster left
print("The real hotspot (seg-06) and its cluster have DISSOLVED — zero significant segments.")

### Side by side · *Lado a lado*
Same reports, same exposure, same statistics — three segmentations, three
different "truths." · *Mismos reportes, misma exposición, misma estadística —
tres segmentaciones, tres "verdades" diferentes.*

In [ ]:
from IPython.display import Markdown

table = [
    "| Segmentation | # units (m) | Significant hotspots |",
    "| --- | ---: | --- |",
    f"| Baseline (as published) | {len(base_rate)} | {', '.join(sorted(sig0)) or '—'} |",
    f"| Split seg-05 into 2 | {len(split_rate)} | {', '.join(sorted(sig1)) or '—'} |",
    f"| Merge 5th St corridor | {len(merge_rate)} | {', '.join(sorted(sig2)) or '—'} |",
]
Markdown("\n".join(table))

---
## What keeps analysts honest · *Qué mantiene honestos a los analistas*

### English
The confidence interval and the FDR correction are real protections — against
*sampling* noise and against *multiple-comparison* luck respectively. Neither can
protect against a **unit of analysis chosen after seeing the data**. Defenses that
do apply:

- **Pre-register the segmentation.** Fix the street-segment definition (e.g. block
  faces split at intersections, as the fixtures and `tools/make_fixtures.py` do)
  *before* computing any hotspot, and never tune it toward a desired result.
- **Report sensitivity to the unit.** Show how the significant set changes under
  reasonable alternative segmentations — the way this notebook just did — rather
  than presenting one as if it were inevitable.
- **Keep the whole comparison set.** Dropping "boring" segments shrinks `m` and
  loosens the FDR threshold; analyze every eligible unit, not a curated subset.
- **Anchor significance to the reports, not the geometry.** A cluster that
  survives only one hand-drawn segmentation is a MAUP artifact, not a finding.

This is the honest core of the whole module: exposure normalization defeats the
*busy-decoy* lie (notebook 02), and a pre-registered, sensitivity-checked unit of
analysis defeats the *re-segmentation* lie (this notebook). Both map back to
threat-model **T4**: make the honest reading the easy reading.

### Español
El intervalo de confianza y la corrección FDR son protecciones reales — contra el
ruido de *muestreo* y contra la suerte de *comparaciones múltiples*,
respectivamente. Ninguno puede proteger contra una **unidad de análisis elegida
después de ver los datos**. Defensas que sí aplican:

- **Pre-registra la segmentación.** Fija la definición de segmento de calle
  *antes* de calcular cualquier punto caliente, y nunca la ajustes hacia un
  resultado deseado.
- **Reporta la sensibilidad a la unidad.** Muestra cómo cambia el conjunto
  significativo bajo segmentaciones alternativas razonables.
- **Conserva todo el conjunto de comparación.** Descartar segmentos "aburridos"
  reduce `m` y afloja el umbral FDR; analiza cada unidad elegible.
- **Ancla la significancia a los reportes, no a la geometría.** Un grupo que
  sobrevive solo a una segmentación dibujada a mano es un artefacto MAUP, no un
  hallazgo.

Este es el núcleo honesto del módulo: la normalización por exposición vence la
mentira del *señuelo concurrido* (cuaderno 02), y una unidad de análisis
pre-registrada y con análisis de sensibilidad vence la mentira de la
*re-segmentación* (este cuaderno). Ambas remiten a **T4** del modelo de amenazas.